In [1]:
from dotenv import load_dotenv
import os
import pandas as pd
import anthropic
import mlflow
from mlflow.metrics import latency
from mlflow.metrics.genai import faithfulness, answer_correctness

In [2]:
# Load environment files and store API keys
load_dotenv()
anthropic_key = os.getenv("ANTHROPIC_KEY")

In [3]:
# Read in dialogue data
file_path = "../../data/Hallucination/dialogue_data.json"
dialogue_data = pd.read_json(file_path, lines=True)

file_path = "../../data/Hallucination/general_data.json"
general_data = pd.read_json(file_path, lines=True)

file_path = "../../data/Hallucination/qa_data.json"
qa_data = pd.read_json(file_path, lines=True)

In [4]:
class Claude():
    """Class to implement Claude model for DeepEval"""
    def __init__(self, model, api_key):
        self.model = model
        self.api_key = api_key
        self.client = anthropic.Client(api_key=self.api_key)

    def load_model(self):
        return self.model

    def generate(self, prompt, max_tokens = 8000):
        response = self.client.messages.create(
            model=self.model,
            max_tokens=max_tokens,
            messages=[
                {"role": "user", "content": prompt}
            ]
        )
        return response.content[0].text

    async def a_generate(self, prompt, max_tokens = 10000):
        
        model = self.load_model()
        
        response = self.client.messages.create(
            model=self.model,
            max_tokens=max_tokens,
            messages=[
                {"role": "user", "content": prompt}
            ]
        )
        return response.content[0].text

    def get_model_name(self):
        return "Claude Model"

In [5]:
# Initialize the Claude model with appropriate settings
claude_model = "claude-3-5-sonnet-20240620"

# Create Claude model
claude_instance = Claude(model=claude_model, api_key=anthropic_key)

# Test functionality
print(claude_instance.generate("Hello, Claude"))

Hello! It's nice to meet you. How can I assist you today?


In [ ]:
# Function to query the LLM with hallucination-aware prompt
def evaluate_hallucination(context, question):

In [13]:
qa_data

,knowledge,question,right_answer,hallucinated_answer
0,Arthur's Magazine (1844–1846) was an American ...,Which magazine was started first Arthur's Maga...,Arthur's Magazine,First for Women was started first.
1,The Oberoi family is an Indian family that is ...,The Oberoi family is part of a hotel company t...,Delhi,The Oberoi family's hotel company is based in ...
2,"Allison Beth ""Allie"" Goertz (born March 2, 199...",Musician and satirist Allie Goertz wrote a son...,President Richard Nixon,"Allie Goertz wrote a song about Milhouse, a po..."
3,"Margaret ""Peggy"" Seeger (born June 17, 1935) i...",What nationality was James Henry Miller's wife?,American,James Henry Miller's wife was British.
4,It is a hygroscopic solid that is highly solu...,Cadmium Chloride is slightly soluble in this c...,alcohol,water with a hint of alcohol
...,...,...,...,...
9995,James Norman Hall (22 April 1887 – 5 July 1951...,Are James Norman Hall and Amiri Baraka from th...,yes,James Norman Hall was French.
9996,Love in the Time of Money is a 2002 American r...,The actress who appeared in the 2002 film Love...,1979,The actress who appeared in the 2002 film Love...
9997,"Ape Escape, known in Japan as Excited Saru Get...",how is Ape Escape and Nicktoons Film Festival ...,shorts,Ape Escape and Nicktoons Film Festival are con...
9998,"An accomplished full-forward, Capper kicked 3...",What position did both Warwick Capper and John...,full forward,Warwick Capper played midfield.


In [6]:
eval_data = pd.DataFrame({
    'inputs': qa_data.question,
    'predictions': qa_data.hallucinated_answer,
    'ground_truth': qa_data.knowledge
})

eval_data = eval_data.head()

In [7]:
from mlflow.metrics.genai import EvaluationExample, faithfulness

# Create a good and bad example for faithfulness in the context of this problem
faithfulness_examples = [
    EvaluationExample(
        input="How do I disable MLflow autologging?",
        output="mlflow.autolog(disable=True) will disable autologging for all functions. In Databricks, autologging is enabled by default. ",
        score=2,
        justification="The output provides a working solution, using the mlflow.autolog() function that is provided in the context.",
        grading_context={
            "context": "mlflow.autolog(log_input_examples: bool = False, log_model_signatures: bool = True, log_models: bool = True, log_datasets: bool = True, disable: bool = False, exclusive: bool = False, disable_for_unsupported_versions: bool = False, silent: bool = False, extra_tags: Optional[Dict[str, str]] = None) → None[source] Enables (or disables) and configures autologging for all supported integrations. The parameters are passed to any autologging integrations that support them. See the tracking docs for a list of supported autologging integrations. Note that framework-specific configurations set at any point will take precedence over any configurations set by this function."
        },
    ),
    EvaluationExample(
        input="How do I disable MLflow autologging?",
        output="mlflow.autolog(disable=True) will disable autologging for all functions.",
        score=5,
        justification="The output provides a solution that is using the mlflow.autolog() function that is provided in the context.",
        grading_context={
            "context": "mlflow.autolog(log_input_examples: bool = False, log_model_signatures: bool = True, log_models: bool = True, log_datasets: bool = True, disable: bool = False, exclusive: bool = False, disable_for_unsupported_versions: bool = False, silent: bool = False, extra_tags: Optional[Dict[str, str]] = None) → None[source] Enables (or disables) and configures autologging for all supported integrations. The parameters are passed to any autologging integrations that support them. See the tracking docs for a list of supported autologging integrations. Note that framework-specific configurations set at any point will take precedence over any configurations set by this function."
        },
    ),
]

faithfulness_metric = faithfulness(model="openai:/gpt-4", examples=faithfulness_examples)

In [8]:
eval_df = pd.DataFrame(
    {
        "questions": [
            "What is MLflow?",
            "How to run mlflow.evaluate()?",
            "How to log_table()?",
            "How to load_table()?",
        ],
        "context": [
            "It's an apple",
            "You put a pin in it",
            "I don't know, try",
            "This is Medford, NY"
        ],
        "results": [
            "It's an apple",
            "You put a pin in it",
            "I don't know, try",
            "This is Medford, NY"
        ]
    }
)

In [9]:
results = mlflow.evaluate(
    data=eval_df,
    #model_type="question-answering",
    #evaluators="default",
    predictions="result",
    extra_metrics=[faithfulness_metric],
    evaluator_config={
        "col_mapping": {
            "inputs": "questions",
            "context": "context",
            "result": "results"
        }
    },
)
print(results.metrics)

2025/03/14 18:32:17 INFO mlflow.models.evaluation.default_evaluator: Testing metrics on first row...
c:\Users\Johnh\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 1/1 [00:01<00:00,  1.31s/it]
c:\Users\Johnh\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\Johnh\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\_core\_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
c:\Users\Johnh\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\_core\fromnumeric.py:4268: RuntimeWarning: Degrees of freedom <= 0 for slice
  return _methods._var(a, a

{'faithfulness/v1/mean': np.float64(nan), 'faithfulness/v1/variance': np.float64(nan)}
